In [2]:
import os
import torch
import librosa
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score

/ihome/lshangguan/zhw227/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- 1. 环境与模型初始化 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
BASE_PATH = "evaluation"
SAMPLE_RATE = 16000
WINDOW_SEC = 10
STRIDE_SEC = 10

# --- AST 灵敏度调节器 ---
SPEECH_THRESHOLD = 0.5  # 概率阈值：建议在 0.3 - 0.7 之间调试
MIN_RMS = 0.01          # 能量门限：低于此值直接跳过深度模型检测，判定为非语音

print(f"🚀 正在初始化评估系统... 设备: {DEVICE}")
extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# --- 2. 更新后的标准答案配置 (Ground Truth) ---
# 0: Non-speech, 1: Speech
files_config = {
    "airport.wav": [0] * 1 + [1] * 200,
    "avenue.wav": [0] * 101,
    "on_the_bus.wav": [0] * 1 + [1] * 400,
    "residential_street.wav": [0] * 201 + [1] * 400,
    "indoor_home.wav": [0] * 451,                  # 新增：全是无意义底噪/环境音
    "restaurant.wav": [0] * 251 + [1] * 400        # 新增：251个环境音 + 400个语音
}

def get_ast_prediction(audio_buffer, threshold):
    """单次 AST 窗口预测逻辑"""
    # 物理层过滤：能量太低直接判定为非语音
    rms = np.sqrt(np.mean(audio_buffer**2))
    if rms < MIN_RMS:
        return 0
        
    inputs = extractor(audio_buffer, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    # 累加 AudioSet 中与人声相关的类目概率
    speech_prob = float(probs[0, [0, 1, 2, 3]].sum())
    
    return 1 if speech_prob >= threshold else 0

# --- 3. 执行批量测试 ---
all_y_true = []
all_y_pred = []
per_file_results = {}

print("\n" + "-"*50)
print(f"{'文件名':<25} | {'总片段数':<10} | {'正在处理...'}")
print("-"*50)

for filename, ground_truth in files_config.items():
    file_path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(file_path):
        print(f"{filename:<25} | {'-':<10} | ❌ 文件不存在")
        continue
    
    # 加载音频（librosa 会自动处理长音频）
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    
    file_preds = []
    for i in range(len(ground_truth)):
        start_idx = i * WINDOW_SEC * SAMPLE_RATE
        end_idx = (i + 1) * WINDOW_SEC * SAMPLE_RATE
        chunk = audio[start_idx:end_idx]
        
        # 针对每个 10s 块进行预测
        pred = get_ast_prediction(chunk, SPEECH_THRESHOLD)
        file_preds.append(pred)
    
    # 统计单文件准确率
    file_acc = accuracy_score(ground_truth, file_preds)
    per_file_results[filename] = file_acc
    print(f"{filename:<25} | {len(ground_truth):<10} | ✅ 准确率: {file_acc:.2%}")

    all_y_true.extend(ground_truth)
    all_y_pred.extend(file_preds)

# --- 4. 汇总统计报告 ---
print("\n" + "="*45)
print("             最终性能评估报告")
print("="*45)

# 计算各项指标
acc = accuracy_score(all_y_true, all_y_pred)
pre = precision_score(all_y_true, all_y_pred, zero_division=0)
rec = recall_score(all_y_true, all_y_pred, zero_division=0)
f1  = f1_score(all_y_true, all_y_pred, zero_division=0)
cm  = confusion_matrix(all_y_true, all_y_pred)

print(f"总体准确率 (Overall Accuracy): {acc:.4f}")
print(f"精确率 (Precision):           {pre:.4f}")
print(f"召回率 (Recall):              {rec:.4f}")
print(f"F1 分数 (F1-Score):           {f1:.4f}")
print("-" * 45)

print("\n混淆矩阵 (Confusion Matrix):")
print(f"{'':<15} 预测:非语音    预测:语音")
print(f"实际:非语音     {cm[0][0]:<12} {cm[0][1]:<12}")
print(f"实际:语音       {cm[1][0]:<12} {cm[1][1]:<12}")

print("\n" + "="*45)

🚀 正在初始化评估系统... 设备: cuda


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 2233.41it/s, Materializing param=classifier.layernorm.weight]                                                    



--------------------------------------------------
文件名                       | 总片段数       | 正在处理...
--------------------------------------------------
airport.wav               | 201        | ✅ 准确率: 99.50%
avenue.wav                | 101        | ✅ 准确率: 100.00%
on_the_bus.wav            | 401        | ✅ 准确率: 95.01%
residential_street.wav    | 601        | ✅ 准确率: 99.33%
indoor_home.wav           | 451        | ✅ 准确率: 100.00%
restaurant.wav            | 651        | ✅ 准确率: 89.25%

             最终性能评估报告
总体准确率 (Overall Accuracy): 0.9605
精确率 (Precision):           0.9509
召回率 (Recall):              0.9829
F1 分数 (F1-Score):           0.9666
---------------------------------------------

混淆矩阵 (Confusion Matrix):
                预测:非语音    预测:语音
实际:非语音     935          71          
实际:语音       24           1376        



In [9]:
import os
import torch
import librosa
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import itertools

# --- 1. 配置与模型初始化 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
BASE_PATH = "evaluation"
SAMPLE_RATE = 16000
WINDOW_SEC = 10

# --- 2. 搜索空间定义 (Grid Search Space) ---
# 你可以根据初步测试结果调整这些范围
threshold_range = np.arange(0.1, 0.95, 0.05)  # 0.1, 0.15, ..., 0.9
rms_range = [0.001, 0.005, 0.01, 0.015, 0.02, 0.03, 0.05]

files_config = {
    "airport.wav": [0] * 1 + [1] * 200,
    "avenue.wav": [0] * 101,
    "on_the_bus.wav": [0] * 1 + [1] * 400,
    "residential_street.wav": [0] * 201 + [1] * 400,
    "indoor_home.wav": [0] * 451,
    "restaurant.wav": [0] * 251 + [1] * 400
}

# --- 3. 预扫描阶段 (提取特征) ---
print(f"🚀 正在加载模型并扫描音频特征... (仅运行一次推理)")
extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# 存储结构: (rms_value, speech_probability, true_label)
precomputed_data = []

for filename, ground_truth in files_config.items():
    file_path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(file_path): continue
    
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    print(f"  正在扫描: {filename}...")
    
    for i in range(len(ground_truth)):
        chunk = audio[i * WINDOW_SEC * SAMPLE_RATE : (i + 1) * WINDOW_SEC * SAMPLE_RATE]
        
        # 计算 RMS
        rms_val = np.sqrt(np.mean(chunk**2))
        
        # 运行 AST 提取概率
        inputs = extractor(chunk, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        speech_prob = float(probs[0, [0, 1, 2, 3]].sum())
        
        precomputed_data.append({
            'rms': rms_val,
            'prob': speech_prob,
            'label': ground_truth[i]
        })

# --- 4. 网格搜索阶段 (纯数学计算，极快) ---
print(f"\n🔍 开始在内存中搜索最佳参数组合 ({len(threshold_range) * len(rms_range)} 组)...")

best_f1 = -1
best_params = {}
results_log = []

y_true = [d['label'] for d in precomputed_data]

for th, rms_th in itertools.product(threshold_range, rms_range):
    # 根据当前参数生成预测结果
    y_pred = []
    for d in precomputed_data:
        if d['rms'] < rms_th:
            y_pred.append(0)
        else:
            y_pred.append(1 if d['prob'] >= th else 0)
    
    current_f1 = f1_score(y_true, y_pred, zero_division=0)
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_params = {
            'threshold': th,
            'min_rms': rms_th,
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'accuracy': accuracy_score(y_true, y_pred)
        }

# --- 5. 输出结果 ---
print("\n" + "="*50)
print("             网格搜索最终结论")
print("="*50)
print(f"✨ 最佳 F1 分数: {best_f1:.4f}")
print(f"🛠️ 建议参数设置:")
print(f"   - SPEECH_THRESHOLD: {best_params['threshold']:.2f}")
print(f"   - MIN_RMS:          {best_params['min_rms']:.4f}")
print("-" * 50)
print(f"📈 在该参数下的表现:")
print(f"   - 准确率 (Acc):    {best_params['accuracy']:.4f}")
print(f"   - 精确率 (Prec):   {best_params['precision']:.4f} (误报控制能力)")
print(f"   - 召回率 (Rec):    {best_params['recall']:.4f} (漏报控制能力)")
print("="*50)

🚀 正在加载模型并扫描音频特征... (仅运行一次推理)


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 2206.37it/s, Materializing param=classifier.layernorm.weight]                                                    


  正在扫描: airport.wav...
  正在扫描: avenue.wav...
  正在扫描: on_the_bus.wav...
  正在扫描: residential_street.wav...
  正在扫描: indoor_home.wav...
  正在扫描: restaurant.wav...

🔍 开始在内存中搜索最佳参数组合 (119 组)...

             网格搜索最终结论
✨ 最佳 F1 分数: 0.9641
🛠️ 建议参数设置:
   - SPEECH_THRESHOLD: 0.50
   - MIN_RMS:          0.0100
--------------------------------------------------
📈 在该参数下的表现:
   - 准确率 (Acc):    0.9576
   - 精确率 (Prec):   0.9507 (误报控制能力)
   - 召回率 (Rec):    0.9779 (漏报控制能力)
